In [ ]:
from pathlib import Path
import pandas as pd

carpeta_actual = Path.cwd()

if carpeta_actual.name == "notebooks":
    carpeta_proyecto = carpeta_actual.parent
else:
    carpeta_proyecto = carpeta_actual

carpeta_datos = carpeta_proyecto / "datos"
carpeta_procesados = carpeta_proyecto / "datos_procesados"

carpeta_procesados.mkdir(
    parents=True,
    exist_ok=True
)

ruta_banco = (
    carpeta_datos
    / "comportamiento_bancario.csv"
)

comportamiento = pd.read_csv(ruta_banco)

print("Archivo cargado desde:", ruta_banco)
comportamiento.head()

,ID_Usuario,Mes_Referencia,Saldo_Promedio_Mensual,Dias_Atraso_Otras_Deudas
0,1,2024-01,12768.11,3
1,1,2024-02,15817.67,3
2,1,2024-03,13138.66,0
3,1,2024-04,19453.13,0
4,1,2024-05,14943.44,59


In [ ]:
print("Filas:", comportamiento.shape[0])
print("Columnas:", comportamiento.shape[1])
print("Usuarios distintos:", comportamiento["ID_Usuario"].nunique())

comportamiento.dtypes

Filas: 661139
Columnas: 4
Usuarios distintos: 28993


,0
ID_Usuario,int64
Mes_Referencia,object
Saldo_Promedio_Mensual,float64
Dias_Atraso_Otras_Deudas,int64


In [ ]:
print("Valores faltantes:")
print(comportamiento.isna().sum())

print("\nFilas duplicadas:", comportamiento.duplicated().sum())

duplicados_usuario_mes = comportamiento.duplicated(
    subset=["ID_Usuario", "Mes_Referencia"]
).sum()

print("Usuario y mes duplicados:", duplicados_usuario_mes)

Valores faltantes:
ID_Usuario                  0
Mes_Referencia              0
Saldo_Promedio_Mensual      0
Dias_Atraso_Otras_Deudas    0
dtype: int64

Filas duplicadas: 0
Usuario y mes duplicados: 0


## Preparación temporal

In [ ]:
ruta_train = carpeta_datos / "solicitudes_train.csv"
solicitudes = pd.read_csv(ruta_train)

comportamiento["Mes_Referencia"] = pd.to_datetime(
    comportamiento["Mes_Referencia"],
    format="%Y-%m",
    errors="coerce"
)

solicitudes["Fecha_Solicitud"] = pd.to_datetime(
    solicitudes["Fecha_Solicitud"],
    format="%Y-%m",
    errors="coerce"
)

print(
    "Historial bancario:",
    comportamiento["Mes_Referencia"].min(),
    "a",
    comportamiento["Mes_Referencia"].max()
)

print(
    "Solicitudes:",
    solicitudes["Fecha_Solicitud"].min(),
    "a",
    solicitudes["Fecha_Solicitud"].max()
)

print("Meses bancarios inválidos:", comportamiento["Mes_Referencia"].isna().sum())
print("Fechas de solicitud inválidas:", solicitudes["Fecha_Solicitud"].isna().sum())

Historial bancario: 2024-01-01 00:00:00 a 2025-12-01 00:00:00
Solicitudes: 2025-01-01 00:00:00 a 2025-12-01 00:00:00
Meses bancarios inválidos: 0
Fechas de solicitud inválidas: 0


In [ ]:
historial_usuario = comportamiento[
    comportamiento["ID_Usuario"] == 12698
].sort_values("Mes_Referencia")

historial_usuario

,ID_Usuario,Mes_Referencia,Saldo_Promedio_Mensual,Dias_Atraso_Otras_Deudas
289693,12698,2024-01-01,3712.29,2
289694,12698,2024-02-01,3749.28,25
289695,12698,2024-03-01,3510.29,3
289696,12698,2024-04-01,3997.29,1
289697,12698,2024-05-01,3304.99,7
289698,12698,2024-06-01,4698.37,1
289699,12698,2024-07-01,3966.91,7
289700,12698,2024-08-01,4273.28,0
289701,12698,2024-09-01,3121.25,5
289702,12698,2024-11-01,3647.96,4


In [ ]:
historial_candidatos = solicitudes[
    ["ID_Solicitud", "ID_Usuario", "Fecha_Solicitud"]
].merge(
    comportamiento,
    on= "ID_Usuario",
    how= "left"
)

historial_candidatos.head()

,ID_Solicitud,ID_Usuario,Fecha_Solicitud,Mes_Referencia,Saldo_Promedio_Mensual,Dias_Atraso_Otras_Deudas
0,1,12698,2025-04-01,2024-01-01,3712.29,2
1,1,12698,2025-04-01,2024-02-01,3749.28,25
2,1,12698,2025-04-01,2024-03-01,3510.29,3
3,1,12698,2025-04-01,2024-04-01,3997.29,1
4,1,12698,2025-04-01,2024-05-01,3304.99,7


In [ ]:
condicion_temporal = (
    historial_candidatos["Mes_Referencia"] < historial_candidatos["Fecha_Solicitud"]
)
historial_valido = historial_candidatos[condicion_temporal].copy()

In [ ]:
historial_valido[
    historial_valido["ID_Solicitud"] == 1
]


,ID_Solicitud,ID_Usuario,Fecha_Solicitud,Mes_Referencia,Saldo_Promedio_Mensual,Dias_Atraso_Otras_Deudas
0,1,12698,2025-04-01,2024-01-01,3712.29,2
1,1,12698,2025-04-01,2024-02-01,3749.28,25
2,1,12698,2025-04-01,2024-03-01,3510.29,3
3,1,12698,2025-04-01,2024-04-01,3997.29,1
4,1,12698,2025-04-01,2024-05-01,3304.99,7
5,1,12698,2025-04-01,2024-06-01,4698.37,1
6,1,12698,2025-04-01,2024-07-01,3966.91,7
7,1,12698,2025-04-01,2024-08-01,4273.28,0
8,1,12698,2025-04-01,2024-09-01,3121.25,5
9,1,12698,2025-04-01,2024-11-01,3647.96,4


In [ ]:
meses_historial = (
    historial_valido
    .groupby("ID_Solicitud")
    .size()
)
meses_historial.head()

,0
ID_Solicitud,
1,14
2,21
3,15
4,18
5,12


In [ ]:
meses_historial.loc[1]

np.int64(14)

In [ ]:
print("Solicitudes originales:", solicitudes["ID_Solicitud"].nunique())
print("Solicitudes con historial previo:", meses_historial.index.nunique())

Solicitudes originales: 32416
Solicitudes con historial previo: 32416


In [ ]:
resumen_historial = (
    historial_valido
    .groupby("ID_Solicitud")
    .agg(
        meses_historial = ("Mes_Referencia", "count"),
        saldo_historial = ("Saldo_Promedio_Mensual", "mean"),
        atraso_prom_hist= ("Dias_Atraso_Otras_Deudas", "mean"),
        atraso_max = ("Dias_Atraso_Otras_Deudas", "max"),
        )
)

resumen_historial.head()



,meses_historial,saldo_historial,atraso_prom_hist,atraso_max
ID_Solicitud,,,,
1,14,3695.225714,4.857143,25
2,21,5976.456667,8.809524,56
3,15,178.574667,20.733333,57
4,18,7589.356667,0.944444,4
5,12,3888.616667,3.750000,32


In [ ]:
resumen_historial_tabla = resumen_historial.reset_index()

solicitudes_modelo = solicitudes.merge(
    resumen_historial_tabla,
    on = ("ID_Solicitud"),
    how = ("left")
)

solicitudes_modelo.head()

,ID_Solicitud,ID_Usuario,Monto_Solicitado,Tasa_Interes,Ingreso_Mensual,Motivo_Prestamo,Fecha_Solicitud,Estado_Final,meses_historial,saldo_historial,atraso_prom_hist,atraso_max
0,1,12698,7900,0.2333,4461,Consumo,2025-04-01,1,14,3695.225714,4.857143,25
1,2,20744,17400,0.2507,3889,Vivienda,2025-12-01,1,21,5976.456667,8.809524,56
2,3,6830,8600,0.2368,4035,Vehiculo,2025-06-01,0,15,178.574667,20.733333,57
3,4,5722,13300,0.2826,4018,Vehiculo,2025-07-01,1,18,7589.356667,0.944444,4
4,5,24719,6600,0.2314,3357,Otros,2025-01-01,1,12,3888.616667,3.750000,32


In [ ]:
ruta_train_procesado = (
    carpeta_procesados
    / "solicitudes_train_procesado.csv"
)

solicitudes_modelo.to_csv(
    ruta_train_procesado,
    index=False
)

print("Archivo guardado en:", ruta_train_procesado)

Archivo guardado en: /content/drive/MyDrive/TP_IA_LaDobleS/datos_procesados/solicitudes_train_procesado.csv
